In [ ]:
import pandas as pd

feature = pd.read_csv(r'data/result4/feature.csv')
feature

,city,0,0_work,0_food,0_indoor_rec,0_outdoor_rec,1_work,1_food,1_indoor_rec,1_outdoor_rec,Fandom,Emotion_regulation,Other,cluster,intensity
0,梅州,0.267135,0.007616,0.010252,0.043351,0.123609,0.024897,0.043644,0.072935,0.342121,0.024605,0.228178,0.075864,4,13170
1,吕梁,0.216399,0.019201,0.007784,0.031396,0.072652,0.023612,0.037883,0.069798,0.261806,0.046445,0.291645,0.137519,0,17040
2,舟山,0.189583,0.002981,0.001754,0.016310,0.139425,0.009821,0.017888,0.025956,0.622589,0.013329,0.124518,0.024377,5,31566
3,朝阳,0.189261,0.006911,0.010101,0.028708,0.044657,0.026050,0.044657,0.070707,0.237108,0.036683,0.310473,0.180223,0,7912
4,广安,0.198830,0.005848,0.007443,0.031898,0.067517,0.023923,0.051568,0.080276,0.355130,0.026582,0.259436,0.090377,1,6443
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271,陇南,0.196838,0.007649,0.006119,0.018868,0.075472,0.030087,0.021928,0.046915,0.405915,0.031617,0.296277,0.059153,1,7059
272,保山,0.233857,0.005236,0.001745,0.030017,0.144852,0.011169,0.016056,0.045724,0.539267,0.013962,0.152531,0.037347,2,11598
273,遂宁,0.242872,0.010692,0.006110,0.061100,0.070774,0.025967,0.035642,0.090631,0.336049,0.037169,0.243890,0.080448,0,7474
274,合肥,0.299861,0.010766,0.007555,0.051656,0.099723,0.028740,0.048288,0.106617,0.219875,0.074320,0.240494,0.111622,4,167171


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from scipy.stats import uniform, randint


feature['cluster'] = feature['cluster'].astype(int)


X = feature[['0', '0_work', '0_food', '0_indoor_rec', '0_outdoor_rec', 
             '1_work', '1_food', '1_indoor_rec', '1_outdoor_rec',
             'Fandom', 'Emotion_regulation', 'Other', 'intensity']]
y = feature['cluster']


In [ ]:


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=77, stratify=y)
xgb_clf = XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), use_label_encoder=False, eval_metric='mlogloss')

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 0.5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0.5, 1)
}

random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='accuracy',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=8
)

random_search.fit(X_train, y_train)

print("最佳参数:", random_search.best_params_)
print("\n训练集得分:", random_search.score(X_train, y_train))
print("测试集得分:", random_search.score(X_test, y_test))

y_pred = random_search.predict(X_test)
print("\n分类报告:\n", classification_report(y_test, y_pred))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
最佳参数: {'colsample_bytree': 0.6125253169822235, 'gamma': 0.42114238729749925, 'learning_rate': 0.1449262400109297, 'max_depth': 4, 'n_estimators': 239, 'reg_alpha': 0.9266588657937942, 'reg_lambda': 1.2272719958564209, 'subsample': 0.7306163075223342}

训练集得分: 0.9318181818181818
测试集得分: 0.7142857142857143

分类报告:
               precision    recall  f1-score   support

           0       0.67      1.00      0.80        14
           1       0.50      0.33      0.40         9
           2       0.78      0.78      0.78         9
           3       1.00      1.00      1.00         2
           4       0.73      0.73      0.73        11
           5       0.86      0.55      0.67        11

    accuracy                           0.71        56
   macro avg       0.75      0.73      0.73        56
weighted avg       0.72      0.71      0.70        56



d:\Anaconda\envs\LLM\lib\site-packages\xgboost\core.py:158: UserWarning: [12:06:37] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
import shap
import matplotlib.pyplot as plt


plt.rcParams.update({
    'font.family': 'Microsoft YaHei',
})

best_model = random_search.best_estimator_

explainer = shap.Explainer(best_model, X)
shap_values = explainer(X) 

n_samples, n_features, n_classes = shap_values.values.shape

for c in range(n_classes):
    print(f"Class {c} beeswarm plot:")
    plt.figure()
    class_shap_values = shap.Explanation(values=shap_values.values[:, :, c],
                                        base_values=shap_values.base_values[:, c],
                                        data=shap_values.data,
                                        feature_names=shap_values.feature_names)
    shap.plots.beeswarm(class_shap_values, show=False, max_display=15)
    # plt.savefig(rf"plot/S/class{c}_beeswarm.png", bbox_inches='tight', dpi=300)
    plt.close()


Class 0 beeswarm plot:
Class 1 beeswarm plot:
Class 2 beeswarm plot:
Class 3 beeswarm plot:
Class 4 beeswarm plot:
Class 5 beeswarm plot:
